# Ad Click-Through and Conversion Analysis

Analyze ad performance event data to study click-through rate (CTR), click-to-conversion rate (CVR), bounce-like post-click abandonment, audience response, and creative effectiveness. The notebook focuses on a common failure mode in paid media: strategies that increase clicks but do not translate into purchases.


## What This Notebook Answers

- Which ad creatives and platforms generate the strongest CTR?
- Which audiences respond with engagement, and which actually convert?
- Where does the biggest funnel leak happen between impression, click, and purchase?
- Which combinations are improving clicks but not conversions?
- What should be scaled, fixed, or deprioritized?


## Dataset And Metric Definitions

This project uses four public CSV files hosted on GitHub:

- `ad_events.csv`: impression, click, purchase, and engagement events
- `ads.csv`: platform, creative type, and targeting settings
- `campaigns.csv`: campaign names, dates, and total budgets
- `users.csv`: audience demographics and interests

Important limitation:

- The dataset does **not** include literal website-session bounce rate.
- For this notebook, **bounce behavior** is defined as **post-click abandonment**:
  `post_click_abandonment = 1 - CVR = 1 - purchases / clicks`
- This keeps the analysis honest: we can study where interest is generated but not completed, without pretending to have page-session analytics.


In [1]:
from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (12, 5)

BASE_URL = "https://raw.githubusercontent.com/prathams0ni/Data-Driven_Social_Media_Marketing_Campaign_Performance_Analysis/main"
DATA_URLS = {
    "events": f"{BASE_URL}/ad_events.csv",
    "ads": f"{BASE_URL}/ads.csv",
    "campaigns": f"{BASE_URL}/campaigns.csv",
    "users": f"{BASE_URL}/users.csv",
}

DAY_ORDER = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
TIME_ORDER = ["Morning", "Afternoon", "Evening", "Night"]
AGE_ORDER = ["18-24", "25-34", "35-44", "45-54", "55+"]

print("Imports and settings loaded.")


Imports and settings loaded.


## Load Source Tables

Load the four source tables and confirm the scale of the analysis. Because this is event-level data, the `ad_events` table is the dominant fact table.


In [2]:
events = pd.read_csv(DATA_URLS["events"])
ads = pd.read_csv(DATA_URLS["ads"])
campaigns = pd.read_csv(DATA_URLS["campaigns"])
users = pd.read_csv(DATA_URLS["users"])

tables = {
    "events": events,
    "ads": ads,
    "campaigns": campaigns,
    "users": users,
}

shape_frame = pd.DataFrame(
    {
        "table": list(tables.keys()),
        "rows": [len(table) for table in tables.values()],
        "columns": [table.shape[1] for table in tables.values()],
    }
)

display(shape_frame)
display(events.head())
display(ads.head())
display(campaigns.head())
display(users.head())


,table,rows,columns
0,events,400000,7
1,ads,200,7
2,campaigns,50,6
3,users,10000,7


,event_id,ad_id,user_id,timestamp,day_of_week,time_of_day,event_type
0,1,197,2359b,2025-07-26 00:19:56,Saturday,Night,Like
1,2,51,f9c67,2025-06-15 08:28:07,Sunday,Morning,Share
2,3,46,5b868,2025-06-27 00:40:02,Friday,Night,Impression
3,4,166,3d440,2025-06-05 19:20:45,Thursday,Evening,Impression
4,5,52,68f1a,2025-07-22 08:30:29,Tuesday,Morning,Impression


,ad_id,campaign_id,ad_platform,ad_type,target_gender,target_age_group,target_interests
0,1,28,Facebook,Video,Female,35-44,"art, technology"
1,2,33,Facebook,Stories,All,25-34,"travel, photography"
2,3,20,Instagram,Carousel,All,25-34,technology
3,4,28,Facebook,Stories,Female,25-34,news
4,5,24,Instagram,Image,Female,25-34,news


,campaign_id,name,start_date,end_date,duration_days,total_budget
0,1,Campaign_1_Launch,2025-05-25,2025-07-23,59,"24,021.3200"
1,2,Campaign_2_Launch,2025-04-16,2025-07-07,82,"79,342.4100"
2,3,Campaign_3_Winter,2025-05-04,2025-06-29,56,"14,343.2500"
3,4,Campaign_4_Summer,2025-06-04,2025-08-08,65,"45,326.6000"
4,5,Campaign_5_Launch,2025-07-11,2025-08-28,48,"68,376.6900"


,user_id,user_gender,user_age,age_group,country,location,interests
0,a2474,Female,24,18-24,United Kingdom,New Mariomouth,"fitness, health"
1,141e5,Male,21,18-24,Germany,Danielsfort,"food, fitness, lifestyle"
2,34db0,Male,27,25-34,Australia,Vincentchester,"fashion, news"
3,20d08,Female,28,25-34,India,Lisaport,"health, news, finance"
4,9e830,Male,28,25-34,United States,Brownmouth,"health, photography, lifestyle"


## Validation And Event Taxonomy

Before computing KPIs, verify duplicates, missingness, and the available event types. The critical check is whether the data actually contains the funnel stages needed for CTR and CVR: `Impression`, `Click`, and `Purchase`.


In [3]:
validation_rows = []
for name, table in tables.items():
    validation_rows.append(
        {
            "table": name,
            "rows": len(table),
            "columns": table.shape[1],
            "duplicate_rows": int(table.duplicated().sum()),
            "missing_cells": int(table.isna().sum().sum()),
        }
    )

validation_report = pd.DataFrame(validation_rows)
event_counts = events["event_type"].value_counts().rename_axis("event_type").reset_index(name="count")

display(validation_report)
display(event_counts)

required_events = {"Impression", "Click", "Purchase"}
missing_events = required_events.difference(set(events["event_type"]))
assert not missing_events, f"Missing required funnel events: {sorted(missing_events)}"

print("Validation passed. Required funnel events are present.")


,table,rows,columns,duplicate_rows,missing_cells
0,events,400000,7,0,0
1,ads,200,7,0,0
2,campaigns,50,6,0,0
3,users,10000,7,0,0


,event_type,count
0,Impression,339812
1,Click,40079
2,Like,12013
3,Comment,4108
4,Purchase,2031
5,Share,1957


Validation passed. Required funnel events are present.


## Build A Unified Event Table

Join ad events to ad metadata, campaign metadata, and user demographics. Then derive helper fields for time analysis and target-match analysis.


In [4]:
events["timestamp"] = pd.to_datetime(events["timestamp"])
campaigns["start_date"] = pd.to_datetime(campaigns["start_date"])
campaigns["end_date"] = pd.to_datetime(campaigns["end_date"])

ads["target_gender"] = ads["target_gender"].fillna("All")
ads["target_age_group"] = ads["target_age_group"].fillna("All")
users["user_gender"] = users["user_gender"].fillna("Unknown")
users["age_group"] = pd.Categorical(
    users["age_group"].fillna("Unknown"),
    categories=AGE_ORDER + ["Unknown"],
    ordered=True,
)

df = (
    events.merge(ads, on="ad_id", how="left")
    .merge(campaigns, on="campaign_id", how="left")
    .merge(users, on="user_id", how="left")
)

df["day_of_week"] = pd.Categorical(df["day_of_week"], categories=DAY_ORDER, ordered=True)
df["time_of_day"] = pd.Categorical(df["time_of_day"], categories=TIME_ORDER, ordered=True)
df["event_date"] = df["timestamp"].dt.normalize()
df["event_month"] = df["timestamp"].dt.to_period("M").astype(str)
df["event_hour"] = df["timestamp"].dt.hour
df["is_target_gender_match"] = df["target_gender"].eq("All") | df["target_gender"].eq(df["user_gender"])
df["is_target_age_match"] = df["target_age_group"].eq("All") | df["target_age_group"].astype(str).eq(df["age_group"].astype(str))
df["is_target_match"] = df["is_target_gender_match"] & df["is_target_age_match"]


def summarize_events(frame, group_cols):
    summary = (
        frame.groupby(group_cols + ["event_type"], observed=True)
        .size()
        .unstack(fill_value=0)
        .reset_index()
    )

    for column in ["Impression", "Click", "Purchase", "Like", "Comment", "Share"]:
        if column not in summary.columns:
            summary[column] = 0

    summary = summary.rename(
        columns={
            "Impression": "impressions",
            "Click": "clicks",
            "Purchase": "purchases",
            "Like": "likes",
            "Comment": "comments",
            "Share": "shares",
        }
    )

    summary["engagements"] = summary[["likes", "comments", "shares"]].sum(axis=1)
    summary["ctr"] = summary["clicks"] / summary["impressions"].replace(0, np.nan)
    summary["cvr"] = summary["purchases"] / summary["clicks"].replace(0, np.nan)
    summary["purchase_rate"] = summary["purchases"] / summary["impressions"].replace(0, np.nan)
    summary["engagement_rate"] = summary["engagements"] / summary["impressions"].replace(0, np.nan)
    summary["post_click_abandonment"] = 1 - summary["cvr"]
    summary["post_click_abandonment"] = summary["post_click_abandonment"].clip(lower=0, upper=1)
    return summary


print(f"Merged event table: {len(df):,} rows x {df.shape[1]} columns")
display(df.head())


Merged event table: 403,967 rows x 30 columns


,event_id,ad_id,user_id,timestamp,day_of_week,time_of_day,event_type,campaign_id,ad_platform,ad_type,target_gender,target_age_group,target_interests,name,start_date,end_date,duration_days,total_budget,user_gender,user_age,age_group,country,location,interests,event_date,event_month,event_hour,is_target_gender_match,is_target_age_match,is_target_match
0,1,197,2359b,2025-07-26 00:19:56,Saturday,Night,Like,9,Facebook,Stories,All,All,"lifestyle, gaming",Campaign_9_Launch,2025-05-25,2025-07-13,49,"40,094.0700",Female,24,18-24,United States,West Shawna,"gaming, food",2025-07-26,2025-07,0,True,True,True
1,2,51,f9c67,2025-06-15 08:28:07,Sunday,Morning,Share,26,Instagram,Carousel,All,18-24,photography,Campaign_26_Winter,2025-04-01,2025-06-17,77,"44,538.8700",Female,30,25-34,United States,Meyersland,"photography, finance",2025-06-15,2025-06,8,True,False,False
2,3,46,5b868,2025-06-27 00:40:02,Friday,Night,Impression,10,Instagram,Carousel,All,35-44,"technology, travel",Campaign_10_Winter,2025-05-17,2025-07-21,65,"19,669.2700",Male,20,18-24,United States,Barrerahaven,"fashion, sports, travel",2025-06-27,2025-06,0,True,False,False
3,4,166,3d440,2025-06-05 19:20:45,Thursday,Evening,Impression,14,Instagram,Image,All,All,fashion,Campaign_14_Summer,2025-04-15,2025-06-04,50,"39,849.9400",Female,18,18-24,United States,Lake Angelaland,"food, art",2025-06-05,2025-06,19,True,True,True
4,5,52,68f1a,2025-07-22 08:30:29,Tuesday,Morning,Impression,2,Instagram,Stories,Female,35-44,"health, lifestyle",Campaign_2_Launch,2025-04-16,2025-07-07,82,"79,342.4100",Male,58,NaN,United Kingdom,Robinsonberg,"finance, lifestyle",2025-07-22,2025-07,8,False,False,False


## Overall Performance Snapshot

Start with the global funnel and response mix before drilling into creative, audience, and timing differences.


In [5]:
overall_counts = df["event_type"].value_counts()
impressions = int(overall_counts.get("Impression", 0))
clicks = int(overall_counts.get("Click", 0))
purchases = int(overall_counts.get("Purchase", 0))
engagements = int(overall_counts.reindex(["Like", "Comment", "Share"], fill_value=0).sum())

overall_kpis = pd.DataFrame(
    {
        "metric": [
            "Impressions",
            "Clicks",
            "Purchases",
            "Engagements",
            "CTR",
            "CVR",
            "Purchase rate",
            "Post-click abandonment",
            "Engagement rate",
        ],
        "value": [
            impressions,
            clicks,
            purchases,
            engagements,
            clicks / impressions,
            purchases / clicks,
            purchases / impressions,
            1 - (purchases / clicks),
            engagements / impressions,
        ],
    }
)

event_mix = overall_counts.rename_axis("event_type").reset_index(name="count")
event_mix["share"] = event_mix["count"] / event_mix["count"].sum()

display(overall_kpis)
display(event_mix)

fig, axes = plt.subplots(1, 2, figsize=(18, 5))
sns.barplot(data=event_mix, x="event_type", y="count", color="#2a9d8f", ax=axes[0])
axes[0].set_title("Event Volume By Event Type")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=20)

sns.barplot(data=event_mix, x="event_type", y="share", color="#e76f51", ax=axes[1])
axes[1].set_title("Event Share By Event Type")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()


,metric,value
0,Impressions,"343,157.0000"
1,Clicks,"40,495.0000"
2,Purchases,"2,050.0000"
3,Engagements,"18,265.0000"
4,CTR,0.1180
5,CVR,0.0506
6,Purchase rate,0.0060
7,Post-click abandonment,0.9494
8,Engagement rate,0.0532


,event_type,count,share
0,Impression,343157,0.8495
1,Click,40495,0.1002
2,Like,12145,0.0301
3,Comment,4142,0.0103
4,Purchase,2050,0.0051
5,Share,1978,0.0049


C:\Users\ahmad\AppData\Local\Temp\ipykernel_54792\3326320497.py:52: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Funnel Behavior And Platform-Level Leakage

The top-level funnel shows where volume disappears. Platform comparison then reveals whether some environments are better at translating attention into purchase intent.


In [6]:
funnel = pd.DataFrame(
    {
        "stage": ["Impression", "Click", "Purchase"],
        "count": [impressions, clicks, purchases],
    }
)
funnel["retention_from_previous"] = funnel["count"] / funnel["count"].shift(1)
funnel["drop_from_previous"] = 1 - funnel["retention_from_previous"]

platform_summary = summarize_events(df, ["ad_platform"]).sort_values("purchases", ascending=False)

display(funnel)
display(platform_summary.round(4))

fig, axes = plt.subplots(1, 2, figsize=(18, 5))
sns.barplot(data=funnel, x="stage", y="count", palette="crest", ax=axes[0])
axes[0].set_title("Overall Funnel Counts")
axes[0].set_xlabel("")

platform_plot = platform_summary.melt(
    id_vars=["ad_platform"],
    value_vars=["ctr", "cvr", "post_click_abandonment"],
    var_name="metric",
    value_name="rate",
)
sns.barplot(data=platform_plot, x="ad_platform", y="rate", hue="metric", ax=axes[1])
axes[1].set_title("Platform CTR, CVR, And Post-Click Abandonment")
axes[1].set_xlabel("")

plt.tight_layout()
plt.show()


,stage,count,retention_from_previous,drop_from_previous
0,Impression,343157,NaN,NaN
1,Click,40495,0.1180,0.8820
2,Purchase,2050,0.0506,0.9494


event_type,ad_platform,clicks,comments,impressions,likes,purchases,shares,engagements,ctr,cvr,purchase_rate,engagement_rate,post_click_abandonment
0,Facebook,25641,2655,218042,7589,1333,1288,11532,0.1176,0.0520,0.0061,0.0529,0.9480
1,Instagram,14854,1487,125115,4556,717,690,6733,0.1187,0.0483,0.0057,0.0538,0.9517


C:\Users\ahmad\AppData\Local\Temp\ipykernel_54792\2292157482.py:16: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=funnel, x="stage", y="count", palette="crest", ax=axes[0])


C:\Users\ahmad\AppData\Local\Temp\ipykernel_54792\2292157482.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Creative Effectiveness

Compare creative formats on both attention and completion. A strong format should not only attract clicks, but also hold enough intent quality to convert those clicks into purchases.


In [7]:
creative_summary = summarize_events(df, ["ad_type"]).sort_values("purchases", ascending=False)
creative_summary["creative_effectiveness_score"] = 100 * (
    0.35 * creative_summary["ctr"].rank(pct=True)
    + 0.35 * creative_summary["cvr"].rank(pct=True)
    + 0.15 * creative_summary["engagement_rate"].rank(pct=True)
    + 0.15 * creative_summary["purchase_rate"].rank(pct=True)
)

display(creative_summary.round(4))

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

creative_score_plot = creative_summary.sort_values("creative_effectiveness_score", ascending=False)
sns.barplot(data=creative_score_plot, x="ad_type", y="creative_effectiveness_score", palette="viridis", ax=axes[0])
axes[0].set_title("Creative Effectiveness Score")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=20)

creative_rate_plot = creative_summary.melt(
    id_vars=["ad_type"],
    value_vars=["ctr", "cvr", "post_click_abandonment"],
    var_name="metric",
    value_name="rate",
)
sns.barplot(data=creative_rate_plot, x="ad_type", y="rate", hue="metric", ax=axes[1])
axes[1].set_title("Creative Tradeoff: Clicks Vs Conversion")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()


event_type,ad_type,clicks,comments,impressions,likes,purchases,shares,engagements,ctr,cvr,purchase_rate,engagement_rate,post_click_abandonment,creative_effectiveness_score
2,Stories,12916,1345,109987,3897,687,653,5895,0.1174,0.0532,0.0062,0.0536,0.9468,82.5000
0,Carousel,10248,1040,87545,3109,525,474,4623,0.1171,0.0512,0.0060,0.0528,0.9488,46.2500
1,Image,10595,1055,89022,3164,494,508,4727,0.1190,0.0466,0.0055,0.0531,0.9534,55.0000
3,Video,6736,702,56603,1975,344,343,3020,0.1190,0.0511,0.0061,0.0534,0.9489,66.2500


C:\Users\ahmad\AppData\Local\Temp\ipykernel_54792\977775878.py:14: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=creative_score_plot, x="ad_type", y="creative_effectiveness_score", palette="viridis", ax=axes[0])


C:\Users\ahmad\AppData\Local\Temp\ipykernel_54792\977775878.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Audience Response

Audience response is wider than conversion alone. This section compares click response, purchase efficiency, and engagement response across age and gender, then surfaces the countries with enough scale to matter.


In [8]:
audience_summary = summarize_events(df, ["age_group", "user_gender"]).sort_values("purchases", ascending=False)
country_summary = summarize_events(df, ["country"])
country_summary = country_summary[country_summary["impressions"] >= country_summary["impressions"].median()]
country_summary = country_summary.sort_values("purchases", ascending=False).head(10)

display(audience_summary.round(4))
display(country_summary[["country", "impressions", "clicks", "purchases", "ctr", "cvr", "engagement_rate"]].round(4))

ctr_heatmap = audience_summary.pivot(index="age_group", columns="user_gender", values="ctr")
cvr_heatmap = audience_summary.pivot(index="age_group", columns="user_gender", values="cvr")

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
sns.heatmap(ctr_heatmap, annot=True, fmt=".3f", cmap="Blues", ax=axes[0])
axes[0].set_title("CTR By Age Group And Gender")

sns.heatmap(cvr_heatmap, annot=True, fmt=".3f", cmap="Greens", ax=axes[1])
axes[1].set_title("CVR By Age Group And Gender")

plt.tight_layout()
plt.show()


event_type,age_group,user_gender,clicks,comments,impressions,likes,purchases,shares,engagements,ctr,cvr,purchase_rate,engagement_rate,post_click_abandonment
4,25-34,Male,9225,942,78590,2791,471,443,4176,0.1174,0.0511,0.0060,0.0531,0.9489
1,18-24,Male,7056,715,59950,2117,363,353,3185,0.1177,0.0514,0.0061,0.0531,0.9486
3,25-34,Female,5813,590,48448,1664,288,279,2533,0.1200,0.0495,0.0059,0.0523,0.9505
0,18-24,Female,4293,415,36588,1301,223,206,1922,0.1173,0.0519,0.0061,0.0525,0.9481
7,35-44,Male,3258,349,28049,1004,155,171,1524,0.1162,0.0476,0.0055,0.0543,0.9524
6,35-44,Female,2047,212,16926,581,126,106,899,0.1209,0.0616,0.0074,0.0531,0.9384
5,25-34,Other,1817,195,15210,532,98,84,811,0.1195,0.0539,0.0064,0.0533,0.9461
2,18-24,Other,1183,127,10206,348,58,72,547,0.1159,0.0490,0.0057,0.0536,0.9510
10,45-54,Male,719,72,5939,230,27,30,332,0.1211,0.0376,0.0045,0.0559,0.9624
8,35-44,Other,580,69,5045,213,26,28,310,0.1150,0.0448,0.0052,0.0614,0.9552


event_type,country,impressions,clicks,purchases,ctr,cvr,engagement_rate
9,United States,103560,12292,656,0.1187,0.0534,0.0531
8,United Kingdom,51761,6027,290,0.1164,0.0481,0.0521
2,Canada,34488,4086,201,0.1185,0.0492,0.0540
5,India,32125,3873,189,0.1206,0.0488,0.0537
4,Germany,28167,3310,152,0.1175,0.0459,0.0533


C:\Users\ahmad\AppData\Local\Temp\ipykernel_54792\3275051036.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Target-Match Analysis

If targeting is doing useful work, people who match the ad's intended age and gender should not just click more, but convert more efficiently.


In [9]:
target_fit = summarize_events(
    df.assign(target_match_label=np.where(df["is_target_match"], "Matched target", "Off-target")),
    ["target_match_label"],
)

display(target_fit.round(4))

fit_plot = target_fit.melt(
    id_vars=["target_match_label"],
    value_vars=["ctr", "cvr", "post_click_abandonment", "engagement_rate"],
    var_name="metric",
    value_name="rate",
)

plt.figure(figsize=(12, 5))
sns.barplot(data=fit_plot, x="target_match_label", y="rate", hue="metric")
plt.title("Matched Vs Off-Target Response Quality")
plt.xlabel("")
plt.show()


event_type,target_match_label,clicks,comments,impressions,likes,purchases,shares,engagements,ctr,cvr,purchase_rate,engagement_rate,post_click_abandonment
0,Matched target,11304,1139,95812,3397,606,547,5083,0.1180,0.0536,0.0063,0.0531,0.9464
1,Off-target,29191,3003,247345,8748,1444,1431,13182,0.1180,0.0495,0.0058,0.0533,0.9505


C:\Users\ahmad\AppData\Local\Temp\ipykernel_54792\3273518942.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Time Patterns

Timing can increase the top of funnel while weakening the bottom. The heatmaps below show whether certain day and time combinations mainly create clicks, or actually carry through to purchases.


In [10]:
day_time_summary = summarize_events(df, ["day_of_week", "time_of_day"])
display(day_time_summary.round(4))

ctr_by_time = day_time_summary.pivot(index="day_of_week", columns="time_of_day", values="ctr")
ctr_by_time = ctr_by_time.reindex(index=DAY_ORDER, columns=TIME_ORDER)

cvr_by_time = day_time_summary.pivot(index="day_of_week", columns="time_of_day", values="cvr")
cvr_by_time = cvr_by_time.reindex(index=DAY_ORDER, columns=TIME_ORDER)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
sns.heatmap(ctr_by_time, annot=True, fmt=".3f", cmap="YlGnBu", ax=axes[0])
axes[0].set_title("CTR By Day And Time")

sns.heatmap(cvr_by_time, annot=True, fmt=".3f", cmap="YlOrRd", ax=axes[1])
axes[1].set_title("CVR By Day And Time")

plt.tight_layout()
plt.show()


event_type,day_of_week,time_of_day,clicks,comments,impressions,likes,purchases,shares,engagements,ctr,cvr,purchase_rate,engagement_rate,post_click_abandonment
0,Monday,Morning,1464,158,12292,456,75,79,693,0.1191,0.0512,0.0061,0.0564,0.9488
1,Monday,Afternoon,1493,141,12253,435,76,65,641,0.1218,0.0509,0.0062,0.0523,0.9491
2,Monday,Evening,1391,164,12255,468,63,89,721,0.1135,0.0453,0.0051,0.0588,0.9547
3,Monday,Night,1375,137,12257,447,80,84,668,0.1122,0.0582,0.0065,0.0545,0.9418
4,Tuesday,Morning,1401,141,12191,424,66,68,633,0.1149,0.0471,0.0054,0.0519,0.9529
5,Tuesday,Afternoon,1454,138,12206,441,61,61,640,0.1191,0.0420,0.0050,0.0524,0.9580
6,Tuesday,Evening,1513,158,12307,443,73,74,675,0.1229,0.0482,0.0059,0.0548,0.9518
7,Tuesday,Night,1463,150,12236,453,70,60,663,0.1196,0.0478,0.0057,0.0542,0.9522
8,Wednesday,Morning,1446,149,12151,461,74,76,686,0.1190,0.0512,0.0061,0.0565,0.9488
9,Wednesday,Afternoon,1491,161,12279,432,57,65,658,0.1214,0.0382,0.0046,0.0536,0.9618


C:\Users\ahmad\AppData\Local\Temp\ipykernel_54792\1040342544.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Click-Heavy But Conversion-Light Diagnosis

This is the most important diagnostic for the project. We aggregate by creative, platform, and audience segment, then look for combinations with above-average click strength but below-average conversion strength.

Interpretation:

- High `CTR` plus low `CVR` means the ad promise is attractive but the downstream experience or message fit is weak.
- High engagement and high abandonment together often signal curiosity without commitment.
- These combinations are where optimization effort should go first.


In [11]:
diagnostic_summary = summarize_events(df, ["ad_type", "ad_platform", "age_group", "user_gender"])
diagnostic_summary = diagnostic_summary[
    (diagnostic_summary["impressions"] >= diagnostic_summary["impressions"].median())
    & (diagnostic_summary["clicks"] >= diagnostic_summary["clicks"].median())
].copy()

diagnostic_summary["ctr_rank"] = diagnostic_summary["ctr"].rank(pct=True)
diagnostic_summary["cvr_rank"] = diagnostic_summary["cvr"].rank(pct=True)
diagnostic_summary["engagement_rank"] = diagnostic_summary["engagement_rate"].rank(pct=True)
diagnostic_summary["abandonment_rank"] = diagnostic_summary["post_click_abandonment"].rank(pct=True)
diagnostic_summary["click_conversion_gap"] = diagnostic_summary["ctr_rank"] - diagnostic_summary["cvr_rank"]
diagnostic_summary["vanity_click_score"] = 100 * (
    0.45 * diagnostic_summary["ctr_rank"]
    + 0.25 * diagnostic_summary["engagement_rank"]
    + 0.30 * diagnostic_summary["abandonment_rank"]
)

click_heavy = diagnostic_summary[
    (diagnostic_summary["ctr_rank"] >= 0.60)
    & (diagnostic_summary["cvr_rank"] <= 0.40)
].sort_values(["click_conversion_gap", "clicks"], ascending=[False, False])

display(
    click_heavy[
        [
            "ad_type",
            "ad_platform",
            "age_group",
            "user_gender",
            "impressions",
            "clicks",
            "purchases",
            "ctr",
            "cvr",
            "post_click_abandonment",
            "engagement_rate",
            "click_conversion_gap",
            "vanity_click_score",
        ]
    ].head(15).round(4)
)

plot_frame = diagnostic_summary.sort_values("clicks", ascending=False).head(40).copy()
plt.figure(figsize=(13, 8))
sns.scatterplot(
    data=plot_frame,
    x="ctr",
    y="cvr",
    size="clicks",
    hue="post_click_abandonment",
    palette="viridis",
    sizes=(50, 500),
)
plt.title("CTR vs CVR For High-Volume Creative / Platform / Audience Segments")

for _, row in plot_frame.nlargest(8, "click_conversion_gap").iterrows():
    plt.text(
        row["ctr"],
        row["cvr"],
        f"{row['ad_type']} | {row['ad_platform']} | {row['age_group']}",
        fontsize=9,
    )

plt.show()


event_type,ad_type,ad_platform,age_group,user_gender,impressions,clicks,purchases,ctr,cvr,post_click_abandonment,engagement_rate,click_conversion_gap,vanity_click_score
37,Image,Instagram,18-24,Male,6608,838,32,0.1268,0.0382,0.9618,0.0540,0.8542,86.9792
88,Video,Instagram,25-34,Male,2354,291,7,0.1236,0.0241,0.9759,0.0612,0.8333,93.4375
66,Stories,Instagram,35-44,Female,1822,257,12,0.1411,0.0467,0.9533,0.0472,0.7292,68.5417
75,Video,Facebook,25-34,Female,6580,802,33,0.1219,0.0411,0.9589,0.0514,0.6458,67.8125
39,Image,Instagram,25-34,Female,5211,642,27,0.1232,0.0421,0.9579,0.0532,0.6458,75.0000
0,Carousel,Facebook,18-24,Female,5135,620,26,0.1207,0.0419,0.9581,0.0499,0.5833,62.5000
1,Carousel,Facebook,18-24,Male,8384,1001,47,0.1194,0.0470,0.9530,0.0512,0.3750,57.6042


C:\Users\ahmad\AppData\Local\Temp\ipykernel_54792\1758847590.py:64: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Ad-Level Leaderboard

Aggregate to the ad level to identify scalable winners and weak assets. This balances click generation against purchase efficiency instead of rewarding raw volume alone.


In [12]:
ad_level = summarize_events(
    df,
    ["ad_id", "campaign_id", "name", "ad_platform", "ad_type", "target_gender", "target_age_group"],
)
ad_level = ad_level[(ad_level["impressions"] >= ad_level["impressions"].median()) & (ad_level["clicks"] > 0)].copy()
ad_level["ad_effectiveness_score"] = 100 * (
    0.30 * ad_level["ctr"].rank(pct=True)
    + 0.35 * ad_level["cvr"].rank(pct=True)
    + 0.20 * ad_level["purchase_rate"].rank(pct=True)
    + 0.15 * (1 - ad_level["post_click_abandonment"].rank(pct=True))
)

leaderboard_columns = [
    "ad_id",
    "name",
    "ad_platform",
    "ad_type",
    "target_gender",
    "target_age_group",
    "impressions",
    "clicks",
    "purchases",
    "ctr",
    "cvr",
    "post_click_abandonment",
    "ad_effectiveness_score",
]

display(ad_level.nlargest(10, "ad_effectiveness_score")[leaderboard_columns].round(4))
display(ad_level.nsmallest(10, "ad_effectiveness_score")[leaderboard_columns].round(4))

top_ads = ad_level.nlargest(12, "ad_effectiveness_score").copy()
top_ads["ad_label"] = "Ad " + top_ads["ad_id"].astype(str)
top_ads = top_ads.sort_values("ad_effectiveness_score")

plt.figure(figsize=(12, 7))
sns.barplot(data=top_ads, x="ad_effectiveness_score", y="ad_label", hue="ad_type", dodge=False)
plt.title("Top Ads By Balanced Effectiveness Score")
plt.xlabel("Effectiveness score")
plt.ylabel("")
plt.show()


event_type,ad_id,name,ad_platform,ad_type,target_gender,target_age_group,impressions,clicks,purchases,ctr,cvr,post_click_abandonment,ad_effectiveness_score
121,122,Campaign_10_Winter,Facebook,Stories,All,35-44,1715,213,17,0.1242,0.0798,0.9202,94.6040
179,180,Campaign_47_Launch,Instagram,Image,Male,18-24,1739,215,18,0.1236,0.0837,0.9163,93.8119
65,66,Campaign_22_Q3,Facebook,Video,All,All,1773,219,16,0.1235,0.0731,0.9269,90.1485
13,14,Campaign_21_Winter,Instagram,Image,Female,35-44,1711,217,14,0.1268,0.0645,0.9355,87.2277
58,59,Campaign_31_Summer,Instagram,Stories,Male,18-24,1717,204,15,0.1188,0.0735,0.9265,86.8812
147,148,Campaign_8_Q3,Facebook,Carousel,Male,All,1718,217,14,0.1263,0.0645,0.9355,86.7327
78,79,Campaign_17_Launch,Instagram,Carousel,Female,25-34,1740,209,15,0.1201,0.0718,0.9282,86.5842
163,164,Campaign_25_Winter,Instagram,Stories,Male,25-34,1737,215,14,0.1238,0.0651,0.9349,84.5050
86,87,Campaign_31_Summer,Instagram,Image,All,18-24,1723,200,14,0.1161,0.0700,0.9300,79.6535
52,53,Campaign_13_Winter,Facebook,Stories,Female,25-34,1760,201,15,0.1142,0.0746,0.9254,78.2673


event_type,ad_id,name,ad_platform,ad_type,target_gender,target_age_group,impressions,clicks,purchases,ctr,cvr,post_click_abandonment,ad_effectiveness_score
25,26,Campaign_42_Summer,Facebook,Image,Female,All,1734,174,5,0.1003,0.0287,0.9713,5.1980
182,183,Campaign_20_Winter,Instagram,Stories,All,All,1756,188,6,0.1071,0.0319,0.9681,10.1485
34,35,Campaign_18_Q3,Instagram,Image,Female,25-34,1733,194,5,0.1119,0.0258,0.9742,12.1287
115,116,Campaign_4_Summer,Facebook,Video,All,All,1743,197,5,0.1130,0.0254,0.9746,13.0198
124,125,Campaign_32_Summer,Instagram,Carousel,Female,18-24,1818,177,7,0.0974,0.0395,0.9605,13.4653
114,115,Campaign_24_Summer,Facebook,Carousel,Female,All,1728,197,4,0.1140,0.0203,0.9797,13.8119
176,177,Campaign_46_Winter,Instagram,Carousel,Female,35-44,1722,191,6,0.1109,0.0314,0.9686,14.0594
106,107,Campaign_25_Winter,Instagram,Carousel,All,25-34,1726,177,7,0.1025,0.0395,0.9605,14.8515
161,162,Campaign_6_Winter,Instagram,Carousel,All,35-44,1721,189,7,0.1098,0.0370,0.9630,17.2772
31,32,Campaign_29_Winter,Instagram,Stories,All,18-24,1752,194,7,0.1107,0.0361,0.9639,17.3762


C:\Users\ahmad\AppData\Local\Temp\ipykernel_54792\1632788162.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Statistical Checks

A distributional check helps confirm whether creative type differences are just noise or show a measurable separation in ad-level conversion performance.


In [13]:
eligible_ad_level = ad_level[ad_level["clicks"] >= ad_level["clicks"].median()].copy()

creative_groups = [
    group["cvr"].dropna().values
    for _, group in eligible_ad_level.groupby("ad_type")
    if len(group["cvr"].dropna()) > 1
]

if len(creative_groups) >= 2:
    kw_stat, kw_p = stats.kruskal(*creative_groups)
else:
    kw_stat, kw_p = np.nan, np.nan

rho, rho_p = stats.spearmanr(
    eligible_ad_level["ctr"],
    eligible_ad_level["post_click_abandonment"],
    nan_policy="omit",
)

stats_summary = pd.DataFrame(
    {
        "test": [
            "Kruskal-Wallis: ad-type CVR separation",
            "Spearman: CTR vs post-click abandonment",
        ],
        "statistic": [kw_stat, rho],
        "p_value": [kw_p, rho_p],
    }
)

corr = eligible_ad_level[
    ["ctr", "cvr", "engagement_rate", "purchase_rate", "post_click_abandonment"]
].corr(method="spearman")

display(stats_summary.round(4))
display(corr.round(3))


,test,statistic,p_value
0,Kruskal-Wallis: ad-type CVR separation,2.7588,0.4303
1,Spearman: CTR vs post-click abandonment,0.0412,0.7743


event_type,ctr,cvr,engagement_rate,purchase_rate,post_click_abandonment
event_type,,,,,
ctr,1.0000,-0.0410,0.2490,0.1250,0.0410
cvr,-0.0410,1.0000,0.0030,0.9780,-1.0000
engagement_rate,0.2490,0.0030,1.0000,0.0420,-0.0030
purchase_rate,0.1250,0.9780,0.0420,1.0000,-0.9780
post_click_abandonment,0.0410,-1.0000,-0.0030,-0.9780,1.0000


## Key Findings

This final section turns the tables above into concise decisions: what is working, what is merely attracting curiosity, and where conversion friction is probably hiding.


In [14]:
best_creative = creative_summary.sort_values("creative_effectiveness_score", ascending=False).iloc[0]
best_platform = platform_summary.sort_values("cvr", ascending=False).iloc[0]
best_audience_pool = audience_summary[audience_summary["impressions"] >= audience_summary["impressions"].median()]
best_audience = best_audience_pool.sort_values("cvr", ascending=False).iloc[0]
best_time_pool = day_time_summary[day_time_summary["clicks"] >= day_time_summary["clicks"].median()]
best_time_slot = best_time_pool.sort_values("cvr", ascending=False).iloc[0]

target_fit_lookup = target_fit.set_index("target_match_label")
if {"Matched target", "Off-target"}.issubset(target_fit_lookup.index):
    match_cvr_lift = target_fit_lookup.loc["Matched target", "cvr"] - target_fit_lookup.loc["Off-target", "cvr"]
else:
    match_cvr_lift = np.nan

lines = []
lines.append(
    f"- Best creative format: {best_creative['ad_type']} with CTR {best_creative['ctr']:.2%}, CVR {best_creative['cvr']:.2%}, and effectiveness score {best_creative['creative_effectiveness_score']:.1f}."
)
lines.append(
    f"- Best conversion platform: {best_platform['ad_platform']} with CVR {best_platform['cvr']:.2%} and post-click abandonment {best_platform['post_click_abandonment']:.2%}."
)
lines.append(
    f"- Highest-converting scaled audience: age {best_audience['age_group']}, {best_audience['user_gender']} with CTR {best_audience['ctr']:.2%} and CVR {best_audience['cvr']:.2%}."
)
lines.append(
    f"- Best conversion time block among scaled slots: {best_time_slot['day_of_week']} {best_time_slot['time_of_day']} with CVR {best_time_slot['cvr']:.2%}."
)

if pd.notna(match_cvr_lift):
    lines.append(
        f"- Target-match matters: matched audiences convert {match_cvr_lift:.2%} better than off-target audiences."
    )

if not click_heavy.empty:
    biggest_gap = click_heavy.iloc[0]
    lines.append(
        f"- Biggest click/conversion mismatch: {biggest_gap['ad_type']} on {biggest_gap['ad_platform']} for age {biggest_gap['age_group']} / {biggest_gap['user_gender']}. CTR is {biggest_gap['ctr']:.2%} but CVR is only {biggest_gap['cvr']:.2%}, implying heavy post-click leakage."
    )

print("\n".join(lines))


- Best creative format: Stories with CTR 11.74%, CVR 5.32%, and effectiveness score 82.5.
- Best conversion platform: Facebook with CVR 5.20% and post-click abandonment 94.80%.
- Highest-converting scaled audience: age 35-44, Female with CTR 12.09% and CVR 6.16%.
- Best conversion time block among scaled slots: Thursday Morning with CVR 5.88%.
- Target-match matters: matched audiences convert 0.41% better than off-target audiences.
- Biggest click/conversion mismatch: Image on Instagram for age 18-24 / Male. CTR is 12.68% but CVR is only 3.82%, implying heavy post-click leakage.


## Recommendations And Common Traps

Recommended actions:

- Scale creatives and platforms that are strong on both CTR and CVR, not CTR alone.
- Prioritize the segments flagged in the mismatch table for landing-page, offer, or message testing.
- Use target-match analysis to reduce wasted clicks from off-target impressions.
- Schedule heavier budget into time blocks that sustain CVR, not just traffic volume.
- Treat high engagement plus high abandonment as a warning sign of curiosity without purchase intent.

Common traps to avoid:

- Rewarding a creative for clicks when its post-click abandonment is still high.
- Treating engagement events as a substitute for purchase efficiency.
- Scaling off-target inventory because it looks cheap at the click layer.
- Calling this metric a true bounce rate. It is a funnel abandonment proxy because the dataset has no session-level landing-page data.
